In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np
model = SentenceTransformer("all-MiniLM-L6-v2")
passage = (
    "The college has many departments and modern facilities. "
    "Students attend lectures and complete assignments. "
) * 100

full_embedding = model.encode(
    passage,
    normalize_embeddings=True
)

print("Maximum sequence length:", model.max_seq_length)
print("Full embedding shape:", full_embedding.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Maximum sequence length: 256
Full embedding shape: (384,)


In [10]:
first_256_tokens = model.tokenizer(
    passage,
    add_special_tokens=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

print("Token count:", first_256_tokens["input_ids"].shape[1])

truncated_text = model.tokenizer.decode(
    first_256_tokens["input_ids"][0],
    skip_special_tokens=True
)

truncated_embedding = model.encode(
    truncated_text,
    normalize_embeddings=True
)

similarity = np.dot(full_embedding, truncated_embedding)

print("Cosine similarity:", round(float(similarity), 4))

Token count: 256
Cosine similarity: 1.0


In [11]:
# Save embeddings to a file
emb = model.encode(passage, normalize_embeddings=True)
np.save("emb.npy", emb)

# Load them back
loaded_emb = np.load("emb.npy")

print("Original shape:", emb.shape)
print("Loaded shape:", loaded_emb.shape)
print("Same values:", np.allclose(emb, loaded_emb))

Original shape: (384,)
Loaded shape: (384,)
Same values: True


In [12]:
from pathlib import Path

print(Path.cwd())
print(Path("emb.npy").exists())

E:\Projects\python-ai-lab
True


In [13]:
import numpy as np

emb = np.load("emb.npy")

print(emb)

[ 1.84209198e-02 -7.89926201e-03  2.63905877e-05  8.50293972e-03
 -4.00935747e-02 -4.22010198e-03 -1.32390596e-02 -2.16182768e-02
  6.04205008e-04  5.54660372e-02  4.00194228e-02 -1.56340431e-02
  2.21793950e-02  6.01500422e-02 -2.26196293e-02 -6.42743856e-02
 -3.05112749e-02 -6.59474656e-02 -2.03686915e-02  2.46096645e-02
  5.24229482e-02 -4.30118330e-02  3.13808881e-02 -2.67962590e-02
 -2.72017755e-02  1.91810762e-03 -3.07168793e-02 -4.37843204e-02
 -3.13587766e-03 -2.39869230e-03 -9.29870363e-03  1.16656214e-01
  4.47689146e-02 -4.38747443e-02  1.37948934e-02  2.29668394e-02
 -5.60841225e-02  3.32779400e-02  1.07702523e-01  2.16213446e-02
 -8.82852226e-02  3.02757844e-02  8.74238238e-02  5.64503968e-02
  2.04200707e-02 -1.01766307e-02 -9.76908728e-02 -3.97049338e-02
  5.38531765e-02  3.93878063e-03 -5.02898032e-03 -6.59641228e-04
 -3.89760770e-02  1.57679163e-03 -6.81760758e-02  1.24808764e-02
  1.24750491e-02 -6.55565038e-02 -7.46625438e-02 -1.63091794e-02
 -1.14550523e-03  8.33028

In [14]:
import numpy as np
from pathlib import Path

def chunk_text(text: str, size: int, overlap: int) -> list[str]:
    words = text.split()
    if size <= 0 or overlap < 0 or overlap >= size:
        return []
    return [
        " ".join(words[i:i + size])
        for i in range(0, len(words), size-overlap) 
    ]
def read_docs(folder: str) -> dict[str,str]:
    folder_path = Path(folder)
    if not folder_path.is_dir():
        raise NotADirectoryError(f"The folder {folder_path} does not exist.")
    files = list(folder_path.glob("*.txt"))
    file_contents = {}
    for file_path in files:
        with open(file_path, "r", encoding="utf-8") as f:
            file_contents[file_path.name] = f.read()
    return file_contents
   
def word_count_counter(text: str) -> list[tuple[str, int]]:
    counts = Counter(text.lower().split())
    return counts.most_common(10)
    
def word_count(text: str) -> list[tuple[str, int]]:
  counts = {}
  for word in text.lower().split():
    counts[word] = counts.get(word, 0) + 1

documents = read_docs("data")

print("Files loaded:", len(documents))
print(documents["college_rules.txt"])

Files loaded: 1
# College Examination and Attendance Rules

Students must maintain a minimum attendance of 75% to be eligible to write semester examinations.

Students who have attendance below the minimum requirement may be denied permission to attend the semester examination, subject to the college's applicable rules.

Section 4.2: Students must submit examination applications before the announced deadline.

Students are not allowed to use mobile phones during examinations. Only approved materials may be brought into the examination hall.

The cumulative grade point average (CGPA) represents a student's overall academic performance across completed semesters.

Examination fees must be paid before the deadline announced by the college administration.

Students should contact the examination cell for clarification about examination schedules, hall tickets, and eligibility requirements.


In [15]:
from pathlib import Path
print(Path.cwd())

E:\Projects\python-ai-lab


In [16]:
text = documents["college_rules.txt"]

chunks = chunk_text(text, size=30, overlap=5)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i}: {chunk}")

Number of chunks: 5

Chunk 0: # College Examination and Attendance Rules Students must maintain a minimum attendance of 75% to be eligible to write semester examinations. Students who have attendance below the minimum requirement may

Chunk 1: below the minimum requirement may be denied permission to attend the semester examination, subject to the college's applicable rules. Section 4.2: Students must submit examination applications before the announced deadline.

Chunk 2: applications before the announced deadline. Students are not allowed to use mobile phones during examinations. Only approved materials may be brought into the examination hall. The cumulative grade point average

Chunk 3: The cumulative grade point average (CGPA) represents a student's overall academic performance across completed semesters. Examination fees must be paid before the deadline announced by the college administration. Students should

Chunk 4: the college administration. Students should contact the exami

In [17]:
chunk_embeddings = model.encode(
    chunks,
    normalize_embeddings=True
)

print("Chunks:", len(chunks))
print("Embedding shape:", chunk_embeddings.shape)

Chunks: 5
Embedding shape: (5, 384)


In [18]:
import numpy as np

def top_k(query, matrix, k):
    matrix_norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    normalized_matrix = matrix / matrix_norms

    normalized_query = query / np.linalg.norm(query)

    scores = normalized_matrix @ normalized_query
    indices = np.argsort(scores)[-k:][::-1]

    return indices, scores[indices]
   
query = "What attendance percentage is required for exams?"

query_embedding = model.encode(
    query,
    normalize_embeddings=True
)

indices, scores = top_k(
    query_embedding,
    chunk_embeddings,
    k=min(3, len(chunks))
)

for idx, score in zip(indices, scores):
    print(f"\nScore: {score:.3f}")
    print(chunks[idx])


Score: 0.720
# College Examination and Attendance Rules Students must maintain a minimum attendance of 75% to be eligible to write semester examinations. Students who have attendance below the minimum requirement may

Score: 0.521
below the minimum requirement may be denied permission to attend the semester examination, subject to the college's applicable rules. Section 4.2: Students must submit examination applications before the announced deadline.

Score: 0.481
applications before the announced deadline. Students are not allowed to use mobile phones during examinations. Only approved materials may be brought into the examination hall. The cumulative grade point average


In [19]:
queries = [
    "What is the minimum attendance required?",
    "Are mobile phones allowed during examinations?",
    "What does CGPA mean?",
    "When must examination fees be paid?",
    "What is explained in Section 4.2?"
]

for query in queries:
    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    )

    indices, scores = top_k(
        query_embedding,
        chunk_embeddings,
        k=min(3, len(chunks))
    )

    print(f"\nQUERY: {query}")

    for idx, score in zip(indices, scores):
        print(f"{score:.3f} | {chunks[idx]}")


QUERY: What is the minimum attendance required?
0.568 | # College Examination and Attendance Rules Students must maintain a minimum attendance of 75% to be eligible to write semester examinations. Students who have attendance below the minimum requirement may
0.553 | below the minimum requirement may be denied permission to attend the semester examination, subject to the college's applicable rules. Section 4.2: Students must submit examination applications before the announced deadline.
0.391 | the college administration. Students should contact the examination cell for clarification about examination schedules, hall tickets, and eligibility requirements.

QUERY: Are mobile phones allowed during examinations?
0.774 | applications before the announced deadline. Students are not allowed to use mobile phones during examinations. Only approved materials may be brought into the examination hall. The cumulative grade point average
0.445 | below the minimum requirement may be denied permissi

In [20]:
paragraph_chunks = [
    paragraph.strip()
    for paragraph in text.split("\n\n")
    if paragraph.strip()
]

paragraph_embeddings = model.encode(
    paragraph_chunks,
    normalize_embeddings=True
)

print("Paragraph chunks:", len(paragraph_chunks))

for i, chunk in enumerate(paragraph_chunks):
    print(f"\nChunk {i}: {chunk}")

Paragraph chunks: 8

Chunk 0: # College Examination and Attendance Rules

Chunk 1: Students must maintain a minimum attendance of 75% to be eligible to write semester examinations.

Chunk 2: Students who have attendance below the minimum requirement may be denied permission to attend the semester examination, subject to the college's applicable rules.

Chunk 3: Section 4.2: Students must submit examination applications before the announced deadline.

Chunk 4: Students are not allowed to use mobile phones during examinations. Only approved materials may be brought into the examination hall.

Chunk 5: The cumulative grade point average (CGPA) represents a student's overall academic performance across completed semesters.

Chunk 6: Examination fees must be paid before the deadline announced by the college administration.

Chunk 7: Students should contact the examination cell for clarification about examination schedules, hall tickets, and eligibility requirements.


In [21]:
query = "When must examination fees be paid?"

query_embedding = model.encode(
    query,
    normalize_embeddings=True
)

indices, scores = top_k(
    query_embedding,
    paragraph_embeddings,
    k=min(3, len(paragraph_chunks))
)

for idx, score in zip(indices, scores):
    print(f"\nScore: {score:.3f} | {paragraph_chunks[idx]}")


Score: 0.819 | Examination fees must be paid before the deadline announced by the college administration.

Score: 0.465 | # College Examination and Attendance Rules

Score: 0.431 | Students must maintain a minimum attendance of 75% to be eligible to write semester examinations.


In [22]:
from sentence_transformers import SentenceTransformer

class Retriever:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.chunks = []
        self.embeddings = None
        
    def index(self, chunks: list[str]):
        if not chunks:
            raise ValueError("Chunks cannot be empty.")
    
        self.chunks = chunks
        self.embeddings = self.model.encode(
            chunks,
            normalize_embeddings=True
        )
    def search(self, query: str, k: int = 3):
        if self.embeddings is None:
            raise ValueError("Index documents before searching.")
    
        if not 1 <= k <= len(self.chunks):
            raise ValueError("k must be between 1 and the number of chunks.")
    
        query_embedding = self.model.encode(
            query,
            normalize_embeddings=True
        )
    
        scores = self.embeddings @ query_embedding
        indices = scores.argsort()[-k:][::-1]
    
        return [
            (int(i), float(scores[i]), self.chunks[i])
            for i in indices
        ]
        
    def save_embeddings(self, path="embeddings.npy"):
        if self.embeddings is None:
            raise ValueError("Index documents before saving.")
    
        np.save(path, self.embeddings)
        
    def load_embeddings(self, path="embeddings.npy", chunks=None):
        self.embeddings = np.load(path)
    
        if chunks is not None:
            if len(chunks) != len(self.embeddings):
                raise ValueError("Chunks and embeddings do not match.")
    
            self.chunks = chunks

In [23]:
retriever = Retriever()

print(retriever.model.max_seq_length)
print(retriever.chunks)
print(retriever.embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

256
[]
None


In [24]:
retriever.index(paragraph_chunks)

print("Chunks:", len(retriever.chunks))
print("Embedding shape:", retriever.embeddings.shape)

Chunks: 8
Embedding shape: (8, 384)


In [25]:
results = retriever.search(
    "When must examination fees be paid?",
    k=3
)

for index, score, chunk in results:
    print(f"\nChunk {index} | Score: {score:.3f}")
    print(chunk)


Chunk 6 | Score: 0.819
Examination fees must be paid before the deadline announced by the college administration.

Chunk 0 | Score: 0.465
# College Examination and Attendance Rules

Chunk 1 | Score: 0.431
Students must maintain a minimum attendance of 75% to be eligible to write semester examinations.


In [26]:
test_cases = [
    {
        "query": "What attendance percentage is required?",
        "expected": "75%"
    },
    {
        "query": "Are mobile phones permitted during exams?",
        "expected": "not allowed to use mobile phones"
    },
    {
        "query": "What does CGPA mean?",
        "expected": "cumulative grade point average"
    },
    {
        "query": "When must examination fees be paid?",
        "expected": "Examination fees must be paid"
    },
    {
        "query": "What does Section 4.2 specify?",
        "expected": "Section 4.2"
    }
]

In [27]:
correct_at_1 = 0

for case in test_cases:
    results = retriever.search(case["query"], k=3)
    top_chunk = results[0][2]

    correct = case["expected"].lower() in top_chunk.lower()
    correct_at_1 += correct

    print(f"\nQuery: {case['query']}")
    print(f"Correct at rank 1: {correct}")
    print(f"Retrieved: {top_chunk}")

print(f"\nHit@1: {correct_at_1 / len(test_cases):.2f}")


Query: What attendance percentage is required?
Correct at rank 1: True
Retrieved: Students must maintain a minimum attendance of 75% to be eligible to write semester examinations.

Query: Are mobile phones permitted during exams?
Correct at rank 1: True
Retrieved: Students are not allowed to use mobile phones during examinations. Only approved materials may be brought into the examination hall.

Query: What does CGPA mean?
Correct at rank 1: True
Retrieved: The cumulative grade point average (CGPA) represents a student's overall academic performance across completed semesters.

Query: When must examination fees be paid?
Correct at rank 1: True
Retrieved: Examination fees must be paid before the deadline announced by the college administration.

Query: What does Section 4.2 specify?
Correct at rank 1: True
Retrieved: Section 4.2: Students must submit examination applications before the announced deadline.

Hit@1: 1.00


In [28]:
retriever.save_embeddings("embeddings.npy")

In [29]:
retriever2 = Retriever()
retriever2.load_embeddings(
    "embeddings.npy",
    chunks=paragraph_chunks
)

print(retriever2.embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(8, 384)


In [30]:
import re

def tokenize_v2(text: str) -> list[str]:
    return re.findall(
        r"\d+(?:\.\d+)+|[a-z0-9]+",
        text.lower()
    )

print(tokenize_v2("Section 4.2 requires 75% attendance"))
samples = [
    "Section 4.2: Attendance is 75%.",
    "CGPA stands for cumulative grade point average.",
    "Students are NOT allowed to use mobile phones!",
    "Section 4.3 has different examination rules."
]

for sentence in samples:
    print(sentence)
    print(tokenize_v2(sentence))
    print()

['section', '4.2', 'requires', '75', 'attendance']
Section 4.2: Attendance is 75%.
['section', '4.2', 'attendance', 'is', '75']

CGPA stands for cumulative grade point average.
['cgpa', 'stands', 'for', 'cumulative', 'grade', 'point', 'average']

Students are NOT allowed to use mobile phones!
['students', 'are', 'not', 'allowed', 'to', 'use', 'mobile', 'phones']

Section 4.3 has different examination rules.
['section', '4.3', 'has', 'different', 'examination', 'rules']



In [31]:
samples = [
    "Section 4.2: Attendance is 75%.",
    "CGPA stands for cumulative grade point average.",
    "Students are NOT allowed to use mobile phones!",
    "Section 4.3 has different examination rules."
]

for sentence in samples:
    print(sentence)
    print(tokenize_v2(sentence))
    print()

Section 4.2: Attendance is 75%.
['section', '4.2', 'attendance', 'is', '75']

CGPA stands for cumulative grade point average.
['cgpa', 'stands', 'for', 'cumulative', 'grade', 'point', 'average']

Students are NOT allowed to use mobile phones!
['students', 'are', 'not', 'allowed', 'to', 'use', 'mobile', 'phones']

Section 4.3 has different examination rules.
['section', '4.3', 'has', 'different', 'examination', 'rules']



In [32]:
docs = [
    "attendance attendance exam",
    "attendance policy rules",
    "section 4.2 exam rules",
    "CGPA exam results",
    "section 4.2 attendance"
]

In [33]:
from collections import Counter, defaultdict

def build_index(docs: list[str]):
    inverted_index = defaultdict(dict)
    doc_lengths = []

    for doc_id, doc in enumerate(docs):
        tokens = tokenize_v2(doc)
        doc_lengths.append(len(tokens))

        term_counts = Counter(tokens)

        for term, count in term_counts.items():
            inverted_index[term][doc_id] = count

    return dict(inverted_index), doc_lengths

In [34]:
index, doc_lengths = build_index(docs)

print("Document lengths:", doc_lengths)

for term in ["attendance", "section", "exam"]:
    print(term, "->", index.get(term, {}))

Document lengths: [3, 3, 4, 3, 3]
attendance -> {0: 2, 1: 1, 4: 1}
section -> {2: 1, 4: 1}
exam -> {0: 1, 2: 1, 3: 1}


In [35]:
import math

N = len(docs)

def idf(term: str) -> float:
    df = len(index.get(term, {}))

    if df == 0:
        return 0.0

    return math.log(N / df)

for term in ["attendance", "exam", "cgpa", "section"]:
    print(term, round(idf(term), 3))

attendance 0.511
exam 0.511
cgpa 1.609
section 0.916


In [36]:
def tfidf_search(query: str, k: int = 3):
    query_terms = tokenize_v2(query)
    scores = {}

    for doc_id, doc in enumerate(docs):
        tokens = tokenize_v2(doc)
        term_counts = Counter(tokens)
        score = 0.0

        for term in query_terms:
            tf = term_counts.get(term, 0)
            score += tf * idf(term)

        scores[doc_id] = score

    ranked = sorted(
        scores.items(),
        key=lambda item: item[1],
        reverse=True
    )

    return [(doc_id, score) for doc_id, score in ranked[:k]]

In [37]:
query = "cgpa results"

for doc_id, score in tfidf_search(query, k=3):
    print(f"Doc {doc_id} | Score: {score:.3f} | {docs[doc_id]}")

Doc 3 | Score: 3.219 | CGPA exam results
Doc 0 | Score: 0.000 | attendance attendance exam
Doc 1 | Score: 0.000 | attendance policy rules


In [38]:
docs = [
    "attendance attendance exam",  # Doc 0
    "attendance policy rules",     # Doc 1
    "section 4.2 exam rules",      # Doc 2
    "CGPA exam results",           # Doc 3
    "section 4.2 attendance"       # Doc 4
]

In [39]:
N = len(docs)

def idf(term: str) -> float:
    df = len(index.get(term, {}))

    if df == 0:
        return 0.0

    return math.log(N / df)


def tfidf_search(query: str, k: int = 3):
    query_terms = tokenize_v2(query)
    scores = {}

    for doc_id, doc in enumerate(docs):
        term_counts = Counter(tokenize_v2(doc))
        score = sum(
            term_counts.get(term, 0) * idf(term)
            for term in query_terms
        )
        scores[doc_id] = score

    ranked = sorted(
        scores.items(),
        key=lambda item: (-item[1], item[0])
    )

    return ranked[:k]

In [40]:
for doc_id, score in tfidf_search("attendance exam", k=3):
    print(f"Doc {doc_id} | {score:.3f} | {docs[doc_id]}")

Doc 0 | 1.532 | attendance attendance exam
Doc 1 | 0.511 | attendance policy rules
Doc 2 | 0.511 | section 4.2 exam rules


In [41]:
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b"
)

X = vectorizer.fit_transform(docs)

def sklearn_tfidf_search(query: str, k: int = 3):
    query_vector = vectorizer.transform([query])
    scores = (X @ query_vector.T).toarray().ravel()

    ranked_ids = sorted(
        range(len(docs)),
        key=lambda i: (-scores[i], i)
    )

    return [(i, float(scores[i])) for i in ranked_ids[:k]]

In [42]:
query = "attendance exam"

print("From scratch:")
for doc_id, score in tfidf_search(query, k=5):
    print(doc_id, round(score, 3), docs[doc_id])

print("\nScikit-learn:")
for doc_id, score in sklearn_tfidf_search(query, k=5):
    print(doc_id, round(score, 3), docs[doc_id])

From scratch:
0 1.532 attendance attendance exam
1 0.511 attendance policy rules
2 0.511 section 4.2 exam rules
3 0.511 CGPA exam results
4 0.511 section 4.2 attendance

Scikit-learn:
0 0.949 attendance attendance exam
1 0.327 attendance policy rules
4 0.306 section 4.2 attendance
3 0.303 CGPA exam results
2 0.271 section 4.2 exam rules


In [43]:
for query in [
    "section 4.2",
    "cgpa results",
    "attendance policy"
]:
    print(f"\nQuery: {query}")
    print("Scratch:", tfidf_search(query, k=3))
    print("Sklearn:", sklearn_tfidf_search(query, k=3))


Query: section 4.2
Scratch: [(2, 1.8325814637483102), (4, 1.8325814637483102), (0, 0.0)]
Sklearn: [(4, 0.9017857303010408), (2, 0.7998677693592811), (0, 0.0)]

Query: cgpa results
Scratch: [(3, 3.2188758248682006), (0, 0.0), (1, 0.0)]
Sklearn: [(3, 0.9037820855345136), (0, 0.0), (1, 0.0)]

Query: attendance policy
Scratch: [(1, 2.120263536200091), (0, 1.0216512475319814), (4, 0.5108256237659907)]
Sklearn: [(1, 0.8306357653043197), (0, 0.49770447617683305), (4, 0.24048868144577265)]


In [44]:
class BM25:
    def __init__(
        self,
        docs: list[str],
        k1: float = 1.5,
        b: float = 0.75
    ):
        if not docs:
            raise ValueError("Documents cannot be empty.")
        if k1 < 0:
            raise ValueError("k1 must be non-negative.")
        if not 0 <= b <= 1:
            raise ValueError("b must be between 0 and 1.")

        self.docs = docs
        self.k1 = k1
        self.b = b

        self.tokenized_docs = [tokenize_v2(doc) for doc in docs]
        self.doc_lengths = [len(doc) for doc in self.tokenized_docs]
        self.N = len(docs)
        self.avgdl = sum(self.doc_lengths) / self.N

        self.term_freqs = []
        self.df = Counter()

        for tokens in self.tokenized_docs:
            counts = Counter(tokens)
            self.term_freqs.append(counts)

            # Each term contributes once to DF per document.
            self.df.update(counts.keys())

        self.idf = {
            term: math.log(
                1 + (self.N - df + 0.5) / (df + 0.5)
            )
            for term, df in self.df.items()
        }

    def score_document(self, query_terms: list[str], doc_id: int) -> float:
        score = 0.0
        counts = self.term_freqs[doc_id]
        doc_length = self.doc_lengths[doc_id]

        for term in query_terms:
            tf = counts.get(term, 0)

            if tf == 0:
                continue

            term_idf = self.idf.get(term, 0.0)

            denominator = (
                tf
                + self.k1 * (
                    1 - self.b
                    + self.b * doc_length / self.avgdl
                )
            )

            score += (
                term_idf
                * tf
                * (self.k1 + 1)
                / denominator
            )

        return score

    def search(self, query: str, k: int = 3):
        if not 1 <= k <= self.N:
            raise ValueError("k must be between 1 and the number of documents.")

        query_terms = tokenize_v2(query)

        scores = [
            self.score_document(query_terms, doc_id)
            for doc_id in range(self.N)
        ]

        ranked_ids = sorted(
            range(self.N),
            key=lambda i: (-scores[i], i)
        )

        return [
            (doc_id, scores[doc_id], self.docs[doc_id])
            for doc_id in ranked_ids[:k]
        ]

In [45]:
bm25 = BM25(docs)

for doc_id, score, text in bm25.search("attendance exam", k=5):
    print(f"Doc {doc_id} | {score:.3f} | {text}")

Doc 0 | 1.340 | attendance attendance exam
Doc 1 | 0.555 | attendance policy rules
Doc 3 | 0.555 | CGPA exam results
Doc 4 | 0.555 | section 4.2 attendance
Doc 2 | 0.484 | section 4.2 exam rules


In [46]:
for query in [
    "attendance",
    "section 4.2",
    "cgpa results",
    "attendance policy",
    "unknownword"
]:
    print(f"\nQuery: {query}")

    for doc_id, score, text in bm25.search(query, k=3):
        print(f"Doc {doc_id} | {score:.3f} | {text}")


Query: attendance
Doc 0 | 0.786 | attendance attendance exam
Doc 1 | 0.555 | attendance policy rules
Doc 4 | 0.555 | section 4.2 attendance

Query: section 4.2
Doc 4 | 1.802 | section 4.2 attendance
Doc 2 | 1.574 | section 4.2 exam rules
Doc 0 | 0.000 | attendance attendance exam

Query: cgpa results
Doc 3 | 2.853 | CGPA exam results
Doc 0 | 0.000 | attendance attendance exam
Doc 1 | 0.000 | attendance policy rules

Query: attendance policy
Doc 1 | 1.981 | attendance policy rules
Doc 0 | 0.786 | attendance attendance exam
Doc 4 | 0.555 | section 4.2 attendance

Query: unknownword
Doc 0 | 0.000 | attendance attendance exam
Doc 1 | 0.000 | attendance policy rules
Doc 2 | 0.000 | section 4.2 exam rules


In [47]:
%pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [48]:
from rank_bm25 import BM25Okapi

reference = BM25Okapi(
    [tokenize_v2(doc) for doc in docs],
    k1=1.5,
    b=0.75
)

query = "attendance exam"
reference_scores = reference.get_scores(tokenize_v2(query))

our_scores = np.array([
    bm25.score_document(tokenize_v2(query), doc_id)
    for doc_id in range(len(docs))
])

print("Our scores:      ", np.round(our_scores, 4))
print("Reference scores:", np.round(reference_scores, 4))

print("Our ranking:", np.argsort(-our_scores))
print("Reference ranking:", np.argsort(-reference_scores))

Our scores:       [1.3404 0.5546 0.4845 0.5546 0.5546]
Reference scores: [0.2823 0.1168 0.102  0.1168 0.1168]
Our ranking: [0 1 3 4 2]
Reference ranking: [0 1 3 4 2]


In [49]:
length_docs = [
    "exam",
    "exam exam exam exam",
    "exam " + " ".join(["policy"] * 30),
    "policy policy policy policy"
]

query = "exam"

for k1 in [0.5, 1.2, 2.0]:
    model = BM25(length_docs, k1=k1, b=0.75)
    scores = [
        model.score_document(tokenize_v2(query), i)
        for i in range(len(length_docs))
    ]

    print(f"\nk1={k1}")
    for i, score in enumerate(scores):
        print(f"Doc {i}: {score:.4f} | length={model.doc_lengths[i]}")


k1=0.5
Doc 0: 0.4602 | length=1
Doc 1: 0.5006 | length=4
Doc 2: 0.2339 | length=31
Doc 3: 0.0000 | length=4

k1=1.2
Doc 0: 0.5645 | length=1
Doc 1: 0.6735 | length=4
Doc 2: 0.1919 | length=31
Doc 3: 0.0000 | length=4

k1=2.0
Doc 0: 0.6485 | length=1
Doc 1: 0.8392 | length=4
Doc 2: 0.1740 | length=31
Doc 3: 0.0000 | length=4


In [50]:
for b in [0.0, 0.75, 1.0]:
    model = BM25(length_docs, k1=1.5, b=b)
    scores = [
        model.score_document(tokenize_v2(query), i)
        for i in range(len(length_docs))
    ]

    print(f"\nb={b}")
    for i, score in enumerate(scores):
        print(f"Doc {i}: {score:.4f} | length={model.doc_lengths[i]}")


b=0.0
Doc 0: 0.3567 | length=1
Doc 1: 0.6485 | length=4
Doc 2: 0.3567 | length=31
Doc 3: 0.0000 | length=4

b=0.75
Doc 0: 0.5995 | length=1
Doc 1: 0.7392 | length=4
Doc 2: 0.1834 | length=31
Doc 3: 0.0000 | length=4

b=1.0
Doc 0: 0.7754 | length=1
Doc 1: 0.7754 | length=4
Doc 2: 0.1578 | length=31
Doc 3: 0.0000 | length=4


In [51]:
from pathlib import Path

text = Path("data/college_rules.txt").read_text(encoding="utf-8")

college_chunks = [
    paragraph.strip()
    for paragraph in text.split("\n\n")
    if paragraph.strip()
]

print("Number of chunks:", len(college_chunks))

Number of chunks: 8


In [52]:
college_bm25 = BM25(college_chunks)

In [53]:
retriever.index(college_chunks)

In [54]:
test_queries = [
    "What is the minimum attendance required?",
    "Are mobile phones permitted during examinations?",
    "What does CGPA mean?",
    "When must examination fees be paid?",
    "What does Section 4.2 specify?"
]

for query in test_queries:
    print(f"\nQUERY: {query}")

    print("BM25:")
    for doc_id, score, chunk in college_bm25.search(query, k=3):
        print(f"  {doc_id} | {score:.3f} | {chunk[:120]}")

    print("Embeddings:")
    for doc_id, score, chunk in retriever.search(query, k=3):
        print(f"  {doc_id} | {score:.3f} | {chunk[:120]}")


QUERY: What is the minimum attendance required?
BM25:
  1 | 2.225 | Students must maintain a minimum attendance of 75% to be eligible to write semester examinations.
  2 | 2.224 | Students who have attendance below the minimum requirement may be denied permission to attend the semester examination, 
  0 | 1.349 | # College Examination and Attendance Rules
Embeddings:
  1 | 0.589 | Students must maintain a minimum attendance of 75% to be eligible to write semester examinations.
  0 | 0.584 | # College Examination and Attendance Rules
  2 | 0.550 | Students who have attendance below the minimum requirement may be denied permission to attend the semester examination, 

QUERY: Are mobile phones permitted during examinations?
BM25:
  4 | 7.346 | Students are not allowed to use mobile phones during examinations. Only approved materials may be brought into the exami
  1 | 1.281 | Students must maintain a minimum attendance of 75% to be eligible to write semester examinations.
  0 | 0.000 | #

In [55]:
def reciprocal_rank_fusion(
    rankings: list[list[int]],
    k: int = 60,
    top_n: int = 5
) -> list[tuple[int, float]]:
    fused_scores = defaultdict(float)

    for ranking in rankings:
        # Ranking is ordered from best to worst.
        for rank, doc_id in enumerate(ranking, start=1):
            fused_scores[doc_id] += 1 / (k + rank)

    fused = sorted(
        fused_scores.items(),
        key=lambda item: (-item[1], item[0])
    )

    return fused[:top_n]

In [56]:
bm25_ranking = [2, 0, 4, 1]
embedding_ranking = [0, 2, 3, 4]

fused = reciprocal_rank_fusion(
    [bm25_ranking, embedding_ranking],
    top_n=5
)

print(fused)

[(0, 0.03252247488101534), (2, 0.03252247488101534), (4, 0.03149801587301587), (3, 0.015873015873015872), (1, 0.015625)]


In [57]:
def hybrid_search(
    query: str,
    bm25_model,
    embedding_retriever,
    k: int = 5,
    candidate_k: int = 10
):
    bm25_results = bm25_model.search(
        query,
        k=min(candidate_k, len(bm25_model.docs))
    )

    embedding_results = embedding_retriever.search(
        query,
        k=min(candidate_k, len(embedding_retriever.chunks))
    )

    # Both rankers must refer to the same chunk list and document IDs.
    bm25_ranking = [doc_id for doc_id, _, _ in bm25_results]
    embedding_ranking = [doc_id for doc_id, _, _ in embedding_results]

    fused = reciprocal_rank_fusion(
        [bm25_ranking, embedding_ranking],
        top_n=min(k, len(bm25_model.docs))
    )

    return [
        (doc_id, score, bm25_model.docs[doc_id])
        for doc_id, score in fused
    ]

In [58]:
evaluation_set = [
    {
        "query": "What is the minimum attendance required?",
        "relevant_ids": {0}
    },
    {
        "query": "Are mobile phones permitted during examinations?",
        "relevant_ids": {1}
    },
]

In [59]:
def hit_at_k(ranked_ids: list[int], relevant_ids: set[int], k: int) -> int:
    return int(bool(set(ranked_ids[:k]) & relevant_ids))


def recall_at_k(
    ranked_ids: list[int],
    relevant_ids: set[int],
    k: int
) -> float:
    if not relevant_ids:
        return 0.0

    found = set(ranked_ids[:k]) & relevant_ids
    return len(found) / len(relevant_ids)


def reciprocal_rank(
    ranked_ids: list[int],
    relevant_ids: set[int]
) -> float:
    for rank, doc_id in enumerate(ranked_ids, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank

    return 0.0

In [60]:
ranked_ids = [4, 2, 0, 1]
relevant_ids = {2, 4}

print("Hit@1:", hit_at_k(ranked_ids, relevant_ids, 1))
print("Hit@3:", hit_at_k(ranked_ids, relevant_ids, 3))
print("Recall@3:", recall_at_k(ranked_ids, relevant_ids, 3))
print("Reciprocal rank:", reciprocal_rank(ranked_ids, relevant_ids))

Hit@1: 1
Hit@3: 1
Recall@3: 1.0
Reciprocal rank: 1.0


In [61]:
def evaluate_ranker(search_fn, evaluation_set, k=3):
    hits = []
    recalls = []
    reciprocal_ranks = []

    for item in evaluation_set:
        results = search_fn(item["query"], k=k)
        ranked_ids = [doc_id for doc_id, *_ in results]
        relevant_ids = item["relevant_ids"]

        hits.append(hit_at_k(ranked_ids, relevant_ids, k))
        recalls.append(recall_at_k(ranked_ids, relevant_ids, k))
        reciprocal_ranks.append(
            reciprocal_rank(ranked_ids, relevant_ids)
        )

    n = len(evaluation_set)

    if n == 0:
        raise ValueError("Evaluation set cannot be empty.")

    return {
        f"Hit@{k}": sum(hits) / n,
        f"Recall@{k}": sum(recalls) / n,
        "MRR": sum(reciprocal_ranks) / n
    }

In [62]:
evaluate_ranker(college_bm25.search, evaluation_set, k=3)

{'Hit@3': 1.0, 'Recall@3': 1.0, 'MRR': 0.41666666666666663}

In [63]:
def chunk_text(
    text: str,
    size: int = 100,
    overlap: int = 0
) -> list[str]:
    if size <= 0:
        raise ValueError("size must be positive")
    if overlap < 0 or overlap >= size:
        raise ValueError("overlap must be >= 0 and < size")

    words = text.split()
    step = size - overlap

    return [
        " ".join(words[i:i + size])
        for i in range(0, len(words), step)
    ]

In [64]:
sample = " ".join(f"word{i}" for i in range(12))

print(chunk_text(sample, size=5, overlap=0))
print(chunk_text(sample, size=5, overlap=2))

['word0 word1 word2 word3 word4', 'word5 word6 word7 word8 word9', 'word10 word11']
['word0 word1 word2 word3 word4', 'word3 word4 word5 word6 word7', 'word6 word7 word8 word9 word10', 'word9 word10 word11']


In [65]:
import re

def sentence_chunk(
    text: str,
    max_sentences: int = 5,
    overlap: int = 1
) -> list[str]:
    if max_sentences <= 0:
        raise ValueError("max_sentences must be positive")
    if overlap < 0 or overlap >= max_sentences:
        raise ValueError("overlap must be >= 0 and < max_sentences")

    text = text.strip()
    if not text:
        return []

    sentences = [
        s.strip()
        for s in re.split(r'(?<=[.!?])\s+', text)
        if s.strip()
    ]

    chunks = []
    step = max_sentences - overlap

    for i in range(0, len(sentences), step):
        chunks.append(" ".join(sentences[i:i + max_sentences]))

    return chunks

In [66]:
sample = (
    "Students must maintain 75% attendance. "
    "Examination fees must be paid before the deadline. "
    "Mobile phones are not allowed during examinations. "
    "CGPA represents academic performance. "
    "Students should contact the examination cell for clarification. "
    "Hall tickets are issued according to college rules."
)

print(sentence_chunk(sample, max_sentences=3, overlap=1))
print(sentence_chunk(sample, max_sentences=5, overlap=1))

['Students must maintain 75% attendance. Examination fees must be paid before the deadline. Mobile phones are not allowed during examinations.', 'Mobile phones are not allowed during examinations. CGPA represents academic performance. Students should contact the examination cell for clarification.', 'Students should contact the examination cell for clarification. Hall tickets are issued according to college rules.']
['Students must maintain 75% attendance. Examination fees must be paid before the deadline. Mobile phones are not allowed during examinations. CGPA represents academic performance. Students should contact the examination cell for clarification.', 'Students should contact the examination cell for clarification. Hall tickets are issued according to college rules.']


In [67]:
def paragraph_chunk(
    text: str,
    min_words: int = 20
) -> list[str]:
    if min_words < 1:
        raise ValueError("min_words must be positive")

    paragraphs = [
        p.strip()
        for p in re.split(r"\n\s*\n", text.strip())
        if p.strip()
    ]

    if not paragraphs:
        return []

    chunks = []
    pending = ""

    for paragraph in paragraphs:
        if pending:
            pending = pending + "\n\n" + paragraph
        else:
            pending = paragraph

        if len(pending.split()) >= min_words:
            chunks.append(pending)
            pending = ""

    if pending:
        if chunks:
            chunks[-1] += "\n\n" + pending
        else:
            chunks.append(pending)

    return chunks

In [68]:
from pathlib import Path

text = Path("data/college_rules.txt").read_text(encoding="utf-8")

In [69]:
strategies = {
    "Fixed 100w, 0 overlap":
        lambda text: chunk_text(text, size=100, overlap=0),

    "Fixed 150w, 30w overlap":
        lambda text: chunk_text(text, size=150, overlap=30),

    "Fixed 250w, 50w overlap":
        lambda text: chunk_text(text, size=250, overlap=50),

    "Sentence 3":
        lambda text: sentence_chunk(text, max_sentences=3, overlap=1),

    "Sentence 5":
        lambda text: sentence_chunk(text, max_sentences=5, overlap=1),

    "Paragraph":
        lambda text: paragraph_chunk(text, min_words=20),
}

In [70]:
def chunk_statistics(chunks: list[str]) -> dict:
    lengths = [len(chunk.split()) for chunk in chunks]

    if not lengths:
        return {
            "chunks": 0,
            "min_words": 0,
            "max_words": 0,
            "avg_words": 0.0
        }

    return {
        "chunks": len(chunks),
        "min_words": min(lengths),
        "max_words": max(lengths),
        "avg_words": round(sum(lengths) / len(lengths), 2)
    }


chunks_by_strategy = {
    name: chunker(text)
    for name, chunker in strategies.items()
}

for name, chunks in chunks_by_strategy.items():
    print(name, chunk_statistics(chunks))

Fixed 100w, 0 overlap {'chunks': 2, 'min_words': 19, 'max_words': 100, 'avg_words': 59.5}
Fixed 150w, 30w overlap {'chunks': 1, 'min_words': 119, 'max_words': 119, 'avg_words': 119.0}
Fixed 250w, 50w overlap {'chunks': 1, 'min_words': 119, 'max_words': 119, 'avg_words': 119.0}
Sentence 3 {'chunks': 4, 'min_words': 29, 'max_words': 55, 'avg_words': 38.25}
Sentence 5 {'chunks': 2, 'min_words': 54, 'max_words': 75, 'avg_words': 64.5}
Paragraph {'chunks': 4, 'min_words': 21, 'max_words': 44, 'avg_words': 29.75}


In [71]:
test_queries = [
    "What is the minimum attendance required?",
    "Are mobile phones permitted during examinations?",
    "What does CGPA mean?",
    "When must examination fees be paid?",
    "What does Section 4.2 specify?"
]

In [72]:
for strategy_name, chunks in chunks_by_strategy.items():
    print(f"\n{'=' * 15} {strategy_name} {'=' * 15}")

    for doc_id, chunk in enumerate(chunks):
        print(f"\nID {doc_id}: {chunk}")


=============== Fixed 100w, 0 overlap ===============

ID 0: # College Examination and Attendance Rules Students must maintain a minimum attendance of 75% to be eligible to write semester examinations. Students who have attendance below the minimum requirement may be denied permission to attend the semester examination, subject to the college's applicable rules. Section 4.2: Students must submit examination applications before the announced deadline. Students are not allowed to use mobile phones during examinations. Only approved materials may be brought into the examination hall. The cumulative grade point average (CGPA) represents a student's overall academic performance across completed semesters. Examination fees must be paid before the deadline announced by

ID 1: the college administration. Students should contact the examination cell for clarification about examination schedules, hall tickets, and eligibility requirements.

=============== Fixed 150w, 30w overlap ==============

In [73]:
relevance_by_strategy = {
    "Fixed 100w, 0 overlap": [
        # Set of relevant chunk IDs for each of the 5 queries
        # Example format: {0}, {1}, {2}, {3}, {4}
    ],
    "Fixed 150w, 30w overlap": [],
    "Fixed 250w, 50w overlap": [],
    "Sentence 3": [],
    "Sentence 5": [],
    "Paragraph": [],
}

In [74]:
def build_strategy_retrievers(chunks):
    bm25_model = BM25(chunks)

    embedding_retriever = Retriever()
    embedding_retriever.index(chunks)

    return bm25_model, embedding_retriever

In [75]:
def hit_at_k(ranked_ids, relevant_ids, k):
    return int(bool(set(ranked_ids[:k]) & relevant_ids))


def reciprocal_rank(ranked_ids, relevant_ids):
    for rank, doc_id in enumerate(ranked_ids, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0

In [76]:
def evaluate_chunk_strategy(
    chunks,
    queries,
    relevant_ids_per_query,
    bm25_model,
    embedding_retriever,
    k=3
):
    if len(queries) != len(relevant_ids_per_query):
        raise ValueError("Each query needs a relevance-label set.")

    results = {}

    for method_name, search_fn in [
        ("BM25", bm25_model.search),
        ("Embeddings", embedding_retriever.search),
    ]:
        hits = []
        rrs = []

        for query, relevant_ids in zip(
            queries, relevant_ids_per_query
        ):
            search_results = search_fn(query, k=k)
            ranked_ids = [doc_id for doc_id, *_ in search_results]

            hits.append(hit_at_k(ranked_ids, relevant_ids, k))
            rrs.append(reciprocal_rank(ranked_ids, relevant_ids))

        results[method_name] = {
            "Hit@3": sum(hits) / len(hits),
            "MRR": sum(rrs) / len(rrs)
        }

    return results

In [77]:
def average_precision(
    ranked_ids: list[int],
    relevant_ids: set[int]
) -> float:
    if not relevant_ids:
        return 0.0

    relevant_found = 0
    precision_sum = 0.0

    for rank, doc_id in enumerate(ranked_ids, start=1):
        if doc_id in relevant_ids:
            relevant_found += 1
            precision_at_rank = relevant_found / rank
            precision_sum += precision_at_rank

    return precision_sum / len(relevant_ids)

In [78]:
ranked_ids = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
relevant_ids = {20, 40, 70}

print(average_precision(ranked_ids, relevant_ids))

0.4761904761904762


In [79]:
ranked_ids = [20, 10, 40, 50]
relevant_ids = {20, 40, 70}

print(average_precision(ranked_ids, relevant_ids))

0.5555555555555555


In [80]:
def mean_average_precision(
    ranked_lists: list[list[int]],
    relevance_sets: list[set[int]]
) -> float:
    if len(ranked_lists) != len(relevance_sets):
        raise ValueError("Each query needs a relevance set.")

    if not ranked_lists:
        raise ValueError("Evaluation set cannot be empty.")

    aps = [
        average_precision(ranked_ids, relevant_ids)
        for ranked_ids, relevant_ids in zip(
            ranked_lists, relevance_sets
        )
    ]

    return sum(aps) / len(aps)

In [81]:
ranked_lists = [
    [20, 10, 40, 50],  # Relevant docs at ranks 1 and 3
    [7, 8, 9, 10]      # One relevant doc at rank 4
]

relevance_sets = [
    {20, 40},
    {10}
]

print("MAP:", mean_average_precision(ranked_lists, relevance_sets))

MAP: 0.5416666666666666


In [82]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [95]:
queries = [
    "What is the minimum attendance required for exams?",
    "When must examination applications be submitted?",
    "Can students use mobile phones during exams?",
    "What does CGPA represent?",
    "When should examination fees be paid?"
]

# Replace each example ID with the correct ID from your printed chunks.
relevance_labels = {
    queries[0]: {0},
    queries[1]: {0},
    queries[2]: {0},
    queries[3]: {0},
    queries[4]: {0}
}

In [96]:
# BM25 candidate retrieval
query = queries[0]

bm25_candidates = college_bm25.search(query, k=5)

print("Before reranking:")
for rank, (doc_id, score, chunk) in enumerate(bm25_candidates, start=1):
    print(f"{rank}. ID={doc_id}, score={score:.3f}: {chunk[:150]}")

Before reranking:
1. ID=1, score=2.225: Students must maintain a minimum attendance of 75% to be eligible to write semester examinations.
2. ID=2, score=2.224: Students who have attendance below the minimum requirement may be denied permission to attend the semester examination, subject to the college's appli
3. ID=7, score=2.056: Students should contact the examination cell for clarification about examination schedules, hall tickets, and eligibility requirements.
4. ID=0, score=1.349: # College Examination and Attendance Rules
5. ID=6, score=0.486: Examination fees must be paid before the deadline announced by the college administration.


In [97]:
reranked_results = rerank(
    query,
    bm25_candidates,
    top_n=min(5, len(bm25_candidates))
)

print("\nAfter reranking:")
for rank, (doc_id, score, chunk) in enumerate(reranked_results, start=1):
    print(f"{rank}. ID={doc_id}, score={score:.3f}: {chunk[:150]}")


After reranking:
1. ID=1, score=7.059: Students must maintain a minimum attendance of 75% to be eligible to write semester examinations.
2. ID=2, score=2.948: Students who have attendance below the minimum requirement may be denied permission to attend the semester examination, subject to the college's appli
3. ID=0, score=-2.072: # College Examination and Attendance Rules
4. ID=7, score=-4.921: Students should contact the examination cell for clarification about examination schedules, hall tickets, and eligibility requirements.
5. ID=6, score=-7.347: Examination fees must be paid before the deadline announced by the college administration.


In [98]:
query = queries[0]

candidates = hybrid_search(
    query,
    college_bm25,
    retriever,
    k=20
)

final_results = rerank(
    query,
    candidates,
    top_n=min(5, len(candidates))
)

for rank, (doc_id, score, chunk) in enumerate(final_results, start=1):
    print(f"{rank}. ID={doc_id}, cross_score={score:.3f}: {chunk[:150]}")

1. ID=1, cross_score=7.059: Students must maintain a minimum attendance of 75% to be eligible to write semester examinations.
2. ID=2, cross_score=2.948: Students who have attendance below the minimum requirement may be denied permission to attend the semester examination, subject to the college's appli
3. ID=0, cross_score=-2.072: # College Examination and Attendance Rules
4. ID=7, cross_score=-4.921: Students should contact the examination cell for clarification about examination schedules, hall tickets, and eligibility requirements.
5. ID=3, cross_score=-7.289: Section 4.2: Students must submit examination applications before the announced deadline.


In [99]:

def evaluate_reranking(
    queries,
    relevance_labels,
    candidate_search_fn,
    top_n=5
):
    if len(queries) != len(relevance_labels):
        raise ValueError("Each query needs relevance labels.")

    before_hits, before_rrs = [], []
    after_hits, after_rrs = [], []

    for query in queries:
        relevant_ids = relevance_labels[query]

        candidates = candidate_search_fn(query)

        if not candidates:
            before_hits.append(0)
            before_rrs.append(0.0)
            after_hits.append(0)
            after_rrs.append(0.0)
            continue

        before_ids = [
            doc_id for doc_id, _, _ in candidates[:top_n]
        ]

        reranked = rerank(
            query,
            candidates,
            top_n=min(top_n, len(candidates))
        )

        after_ids = [
            doc_id for doc_id, _, _ in reranked
        ]

        before_hits.append(
            hit_at_k(before_ids, relevant_ids, top_n)
        )
        before_rrs.append(
            reciprocal_rank(before_ids, relevant_ids)
        )

        after_hits.append(
            hit_at_k(after_ids, relevant_ids, top_n)
        )
        after_rrs.append(
            reciprocal_rank(after_ids, relevant_ids)
        )

    n = len(queries)

    if n == 0:
        raise ValueError("Queries cannot be empty.")

    return {
        "Before reranking": {
            "Hit@5": sum(before_hits) / n,
            "MRR": sum(before_rrs) / n
        },
        "After reranking": {
            "Hit@5": sum(after_hits) / n,
            "MRR": sum(after_rrs) / n
        }
    }


In [100]:
def get_hybrid_candidates(query):
    return hybrid_search(
        query,
        college_bm25,
        retriever,
        k=20
    )

In [101]:
results = evaluate_reranking(
    queries,
    relevance_labels,
    get_hybrid_candidates,
    top_n=5
)

print(results)

{'Before reranking': {'Hit@5': 1.0, 'MRR': 0.31333333333333335}, 'After reranking': {'Hit@5': 0.8, 'MRR': 0.24666666666666665}}


In [91]:
print(relevance_labels)
print(type(next(iter(relevance_labels.values()))))

{'What is the minimum attendance required for exams?': {0}, 'When must examination applications be submitted?': {0}, 'Can students use mobile phones during exams?': {0}, 'What does CGPA represent?': {0}, 'When should examination fees be paid?': {0}}
<class 'set'>


In [92]:
relevance_labels = {
    "What is the minimum attendance required for exams?": {2},
    "When must examination applications be submitted?": {3},
    "Can students use mobile phones during exams?": {4},
}

In [131]:
results = evaluate_reranking(
    queries,
    relevance_labels,
    get_hybrid_candidates,
    top_n=5
)

ValueError: Each query needs relevance labels.

In [132]:
relevant_ids = relevance_labels[query]

In [117]:
results = evaluate_reranking(
    queries,
    relevance_labels,
    get_hybrid_candidates,
    top_n=5
)

ValueError: Each query needs relevance labels.

In [102]:
print("Number of queries:", len(queries))
print("Number of relevance labels:", len(relevance_labels))
print("\nQueries:")
for q in queries:
    print(q)

print("\nRelevance label keys:")
for q in relevance_labels:
    print(q)

Number of queries: 5
Number of relevance labels: 5

Queries:
What is the minimum attendance required for exams?
When must examination applications be submitted?
Can students use mobile phones during exams?
What does CGPA represent?
When should examination fees be paid?

Relevance label keys:
What is the minimum attendance required for exams?
When must examination applications be submitted?
Can students use mobile phones during exams?
What does CGPA represent?
When should examination fees be paid?


In [103]:
relevance_labels = {
    queries[0]: {0},
    queries[1]: {1},
    queries[2]: {2},
    queries[3]: {3},
    queries[4]: {4}
}

In [104]:
assert len(queries) == len(relevance_labels)
assert all(q in relevance_labels for q in queries)

results = evaluate_reranking(
    queries,
    relevance_labels,
    get_hybrid_candidates,
    top_n=5
)

In [105]:
for i, q in enumerate(queries):
    print(i, q)

0 What is the minimum attendance required for exams?
1 When must examination applications be submitted?
2 Can students use mobile phones during exams?
3 What does CGPA represent?
4 When should examination fees be paid?


In [106]:
print("Queries:", len(queries))
print("Relevance labels:", len(relevance_labels))

for q, ids in relevance_labels.items():
    print(q, "->", ids)

Queries: 5
Relevance labels: 5
What is the minimum attendance required for exams? -> {0}
When must examination applications be submitted? -> {1}
Can students use mobile phones during exams? -> {2}
What does CGPA represent? -> {3}
When should examination fees be paid? -> {4}


In [107]:
results = evaluate_reranking(
    queries,
    relevance_labels,
    get_hybrid_candidates,
    top_n=5
)

In [108]:
print(relevance_labels)

for q in queries:
    labels = relevance_labels[q]
    print(repr(q), repr(labels), type(labels))

{'What is the minimum attendance required for exams?': {0}, 'When must examination applications be submitted?': {1}, 'Can students use mobile phones during exams?': {2}, 'What does CGPA represent?': {3}, 'When should examination fees be paid?': {4}}
'What is the minimum attendance required for exams?' {0} <class 'set'>
'When must examination applications be submitted?' {1} <class 'set'>
'Can students use mobile phones during exams?' {2} <class 'set'>
'What does CGPA represent?' {3} <class 'set'>
'When should examination fees be paid?' {4} <class 'set'>


In [109]:
relevant_ids = relevance_labels[query]

In [110]:

for query in queries:
    relevant_ids = relevance_labels[query]
    candidates = get_hybrid_candidates(query)

    reranked = rerank(
        query,
        candidates,
        top_n=min(5, len(candidates))
    )

    print(f"\nQuery: {query}")
    print(f"Relevant chunk IDs: {relevant_ids}")

    print("\nBefore reranking:")
    for rank, (doc_id, score, text) in enumerate(candidates[:5], start=1):
        marker = " ✓ RELEVANT" if doc_id in relevant_ids else ""
        print(f"{rank}. ID={doc_id}, score={score:.4f}{marker}")
        print(f"   {text[:180]}")

    print("\nAfter reranking:")
    for rank, (doc_id, score, text) in enumerate(reranked, start=1):
        marker = " ✓ RELEVANT" if doc_id in relevant_ids else ""
        print(f"{rank}. ID={doc_id}, score={score:.4f}{marker}")
        print(f"   {text[:180]}")



Query: What is the minimum attendance required for exams?
Relevant chunk IDs: {0}

Before reranking:
1. ID=1, score=0.0328
   Students must maintain a minimum attendance of 75% to be eligible to write semester examinations.
2. ID=2, score=0.0320
   Students who have attendance below the minimum requirement may be denied permission to attend the semester examination, subject to the college's applicable rules.
3. ID=0, score=0.0318 ✓ RELEVANT
   # College Examination and Attendance Rules
4. ID=7, score=0.0310
   Students should contact the examination cell for clarification about examination schedules, hall tickets, and eligibility requirements.
5. ID=3, score=0.0308
   Section 4.2: Students must submit examination applications before the announced deadline.

After reranking:
1. ID=1, score=7.0590
   Students must maintain a minimum attendance of 75% to be eligible to write semester examinations.
2. ID=2, score=2.9476
   Students who have attendance below the minimum requirement may be 

In [111]:
relevance_labels = {
    queries[0]: {1},  # Attendance: 75%
    queries[1]: {3},  # Examination application deadline
    queries[2]: {4},  # Mobile phone restriction
    queries[3]: {5},  # CGPA definition
    queries[4]: {6},  # Examination fee deadline
}

In [112]:
results = evaluate_reranking(
    queries,
    relevance_labels,
    get_hybrid_candidates,
    top_n=5
)

results

{'Before reranking': {'Hit@5': 1.0, 'MRR': 1.0},
 'After reranking': {'Hit@5': 1.0, 'MRR': 1.0}}

In [114]:
%pip install -U groq python-dotenv

   ---------------------------------------- 0.0/143.8 kB ? eta -:--:--
   -- ------------------------------------- 10.2/143.8 kB ? eta -:--:--
   ----------------- ---------------------- 61.4/143.8 kB 1.1 MB/s eta 0:00:01
   ---------------------------------------- 143.8/143.8 kB 1.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [117]:

import os
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

load_dotenv(Path.cwd() / ".env")

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY not found. Check your .env file and notebook path."
    )

client = Groq(api_key=api_key)

MODEL_NAME = "openai/gpt-oss-120b"

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Explain RAG in two sentences."}
    ],
    max_completion_tokens=150
)

print(response.choices[0].message.content)


Retrieval‑Augmented Generation (RAG) combines a large language model with an external knowledge base: the model first queries the database (or search engine) to fetch relevant documents, then conditions its generation on both the original prompt and the retrieved text. This lets the system produce up‑to‑date, fact‑rich responses while still leveraging the model’s fluency and reasoning abilities.


In [116]:

models = client.models.list()

for model in models.data:
    if getattr(model, "active", True):
        print(model.id)


canopylabs/orpheus-arabic-saudi
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3-turbo
openai/gpt-oss-120b
whisper-large-v3
qwen/qwen3.8-27b
canopylabs/orpheus-v1-english
allam-2-7b
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-safeguard-20b


In [118]:

def generate_answer(query, reranked_results):
    if not reranked_results:
        return {
            "answer": "I cannot answer this question because no relevant context was retrieved.",
            "citations": []
        }

    context_parts = []

    for chunk_id, score, text in reranked_results:
        context_parts.append(
            f"[Source: college_rules.txt, chunk {chunk_id}]\n{text}"
        )

    context = "\n\n".join(context_parts)

    system_prompt = """
You are a document question-answering assistant.

Rules:
1. Answer using ONLY the supplied context.
2. Do not use outside knowledge or invent facts.
3. If the context does not contain enough information, say:
   "I cannot answer this from the provided documents."
4. Cite every factual answer using the exact source labels provided,
   such as [college_rules.txt, chunk 1].
5. Be concise and clear.
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": (
                    f"Question: {query}\n\n"
                    f"Context:\n{context}\n\n"
                    "Answer the question using the rules above."
                )
            }
        ],
        temperature=0.1,
        max_completion_tokens=300
    )

    answer = response.choices[0].message.content

    citations = [
        {
            "source": f"college_rules.txt, chunk {chunk_id}",
            "text": text
        }
        for chunk_id, score, text in reranked_results
        if f"chunk {chunk_id}" in answer
    ]

    return {
        "answer": answer,
        "citations": citations
    }


In [119]:

query = "What is the minimum attendance required for exams?"

candidates = get_hybrid_candidates(query)

reranked_results = rerank(
    query,
    candidates,
    top_n=min(5, len(candidates))
)

result = generate_answer(query, reranked_results)

print("Answer:")
print(result["answer"])

print("\nCitations:")
for citation in result["citations"]:
    print(citation["source"])


Answer:
The minimum attendance required to be eligible for semester examinations is **75%**【college_rules.txt, chunk 1】.

Citations:
college_rules.txt, chunk 1


In [120]:

query = "What time does the college library close?"

candidates = get_hybrid_candidates(query)

reranked_results = rerank(
    query,
    candidates,
    top_n=min(5, len(candidates))
)

result = generate_answer(query, reranked_results)

print("Answer:")
print(result["answer"])

print("\nCitations:")
print(result["citations"])


Answer:
I cannot answer this from the provided documents.

Citations:
[]


In [121]:

day5_results = []

for query in queries:
    relevant_ids = relevance_labels[query]

    # Stage 1: Hybrid retrieval
    candidates = get_hybrid_candidates(query)

    # Before reranking: evaluate the original top 5
    before_ids = [doc_id for doc_id, _, _ in candidates[:5]]

    # Stage 2: Cross-encoder reranking
    reranked = rerank(
        query,
        candidates,
        top_n=min(5, len(candidates))
    )

    after_ids = [doc_id for doc_id, _, _ in reranked]

    # Stage 3: Generate a grounded answer
    result = generate_answer(query, reranked)

    day5_results.append({
        "query": query,
        "before_ids": before_ids,
        "after_ids": after_ids,
        "before_hit": hit_at_k(before_ids, relevant_ids, 5),
        "after_hit": hit_at_k(after_ids, relevant_ids, 5),
        "before_rr": reciprocal_rank(before_ids, relevant_ids),
        "after_rr": reciprocal_rank(after_ids, relevant_ids),
        "answer": result["answer"],
        "citations": result["citations"]
    })

    print("=" * 80)
    print("QUESTION:", query)
    print("\nBEFORE RERANKING:", before_ids)
    print("AFTER RERANKING: ", after_ids)
    print("\nANSWER:", result["answer"])
    print("CITATIONS:", [c["source"] for c in result["citations"]])


QUESTION: What is the minimum attendance required for exams?

BEFORE RERANKING: [1, 2, 0, 7, 3]
AFTER RERANKING:  [1, 2, 0, 7, 3]

ANSWER: The minimum attendance required to be eligible for semester examinations is **75%**【college_rules.txt, chunk 1】.
CITATIONS: ['college_rules.txt, chunk 1']
QUESTION: When must examination applications be submitted?

BEFORE RERANKING: [3, 6, 1, 7, 0]
AFTER RERANKING:  [3, 6, 1, 7, 4]

ANSWER: Examination applications must be submitted before the announced deadline. 【college_rules.txt, chunk 3】
CITATIONS: ['college_rules.txt, chunk 3']
QUESTION: Can students use mobile phones during exams?

BEFORE RERANKING: [4, 7, 1, 3, 0]
AFTER RERANKING:  [4, 7, 3, 1, 0]

ANSWER: No. Students are not allowed to use mobile phones during examinations. 【college_rules.txt, chunk 4】
CITATIONS: ['college_rules.txt, chunk 4']
QUESTION: What does CGPA represent?

BEFORE RERANKING: [5, 0, 1, 7, 2]
AFTER RERANKING:  [5, 0, 1, 7, 2]

ANSWER: The CGPA (cumulative grade point av

In [122]:

n = len(day5_results)

if n == 0:
    raise ValueError("No results found. Run Step 1 first.")

before_hit = sum(r["before_hit"] for r in day5_results) / n
after_hit = sum(r["after_hit"] for r in day5_results) / n

before_mrr = sum(r["before_rr"] for r in day5_results) / n
after_mrr = sum(r["after_rr"] for r in day5_results) / n

print(f"{'Metric':<12} {'Before':>10} {'After':>10}")
print("-" * 34)
print(f"{'Hit@5':<12} {before_hit:>10.3f} {after_hit:>10.3f}")
print(f"{'MRR':<12} {before_mrr:>10.3f} {after_mrr:>10.3f}")


Metric           Before      After
----------------------------------
Hit@5             1.000      1.000
MRR               1.000      1.000


In [123]:

refusal_query = "What time does the college library close?"

refusal_candidates = get_hybrid_candidates(refusal_query)

refusal_reranked = rerank(
    refusal_query,
    refusal_candidates,
    top_n=min(5, len(refusal_candidates))
) if refusal_candidates else []

refusal_result = generate_answer(refusal_query, refusal_reranked)

print("Question:", refusal_query)
print("Answer:", refusal_result["answer"])
print("Citations:", refusal_result["citations"])

refused = (
    "cannot answer" in refusal_result["answer"].lower()
    or "not enough information" in refusal_result["answer"].lower()
    or "not provided" in refusal_result["answer"].lower()
)

print("Refusal detected:", refused)


Question: What time does the college library close?
Answer: I cannot answer this from the provided documents.
Citations: []
Refusal detected: True
